<a href="https://colab.research.google.com/github/Mohamedboukerche22/simple-llm/blob/main/pytorch/simple_pytorch_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [78]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader , TensorDataset
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from matplotlib import pyplot as plt

In [68]:
X , y = load_breast_cancer(return_X_y=True)
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [69]:
X_train_scaled_tensor = torch.from_numpy(X_train_scaled).float()
X_test_scaled_tensor = torch.from_numpy(X_test_scaled).float()
#X_train_scaled_tensor , X_test_scaled_tensor
y_train_tensor  = torch.from_numpy(y_train).float().unsqueeze(1)
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)

In [70]:
train_dataset = TensorDataset(X_train_scaled_tensor , y_train_tensor)

In [71]:
train_loader = DataLoader(train_dataset , batch_size=32 , shuffle=True)

In [72]:
# Creating nn
class BCNet(nn.Module):

  def __init__(self):
    super(BCNet,self).__init__()

    self.fc1 = nn.Linear(30,64)
    self.fc2 = nn.Linear(64,32)
    self.fc3 = nn.Linear(32,1)


  def forward(self,x):
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = F.sigmoid(self.fc3(x))

    return x

In [73]:
criterion = nn.BCELoss()
model = BCNet()
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [83]:
epoches = 1000
for epoch in range(epoches):
  model.train()
  running_loss = 0.0

  for x_batch , y_batch  in train_loader:
    optimizer.zero_grad()

    preds = model(x_batch)
    loss = criterion(preds , y_batch)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()

  print(f"running epoch {epoch+1} : loss = {running_loss / len(train_loader)}")

running epoch 1 : loss = 0.00026229518844047563
running epoch 2 : loss = 0.0002475769716738796
running epoch 3 : loss = 0.0002875187810635301
running epoch 4 : loss = 0.00024934164927496266
running epoch 5 : loss = 0.00023619072635483462
running epoch 6 : loss = 0.0002691839533326856
running epoch 7 : loss = 0.0002220454722115998
running epoch 8 : loss = 0.0002338387240039689
running epoch 9 : loss = 0.000213827374864195
running epoch 10 : loss = 0.00020962478265573736
running epoch 11 : loss = 0.000203745013398778
running epoch 12 : loss = 0.00019880556008047278
running epoch 13 : loss = 0.0002293217025908234
running epoch 14 : loss = 0.00021826275163524164
running epoch 15 : loss = 0.00019182877301015348
running epoch 16 : loss = 0.00019344526265285822
running epoch 17 : loss = 0.00018486770036171645
running epoch 18 : loss = 0.00017428120707260557
running epoch 19 : loss = 0.00019613833623755757
running epoch 20 : loss = 0.00016860947353052324
running epoch 21 : loss = 0.00016776312

In [88]:
with torch.no_grad():
  model.eval()
  preds = model(X_test_scaled_tensor)
  loss = criterion(preds , y_test_tensor).item()

  accuracy = ((preds >= 0.5 ) == y_test_tensor).float().mean().item()

In [87]:
accuracy

0.9736841917037964